# Dataset And Manifest Review

## Simple English First
This notebook reviews how the project currently stores dataset information. A manifest is just a table that lists image paths, labels, dataset IDs, and split assignments.

The goal is to answer:
- which manifests exist now
- which ones are populated
- whether split information is present
- what is still pending for D2 and D3


## Technical View
The canonical manifest convention uses `data/manifests/*.csv` and resolves dataset-root-relative paths against roots defined in `configs/data/datasets.yaml`.

In [1]:
from pathlib import Path
import pandas as pd
import yaml

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the TyreVisionX repo root from the current working directory.')

repo_root = find_repo_root()
print('Repo root:', repo_root)
cfg = yaml.safe_load((repo_root / 'configs/data/datasets.yaml').read_text(encoding='utf-8'))
cfg

Repo root: /Users/ritik/Documents/Project TDA/TyreVisionX


{'label_mapping': {'good': 0, 'defect': 1},
 'use_datasets': ['D1'],
 'splits': {'train': 0.7, 'val': 0.15, 'test': 0.15, 'split_column': 'split'},
 'paths': {'D1': {'root': 'data/D1_tyrenet',
   'manifest': 'data/manifests/D1_tyrenet_manifest.csv'},
  'D2': {'root': 'data/D2_tire_crack',
   'manifest': 'data/manifests/D2_crack_manifest.csv'},
  'D3': {'root': 'data/D3_tyre_quality',
   'manifest': 'data/manifests/D3_quality_manifest.csv'}},
 'combine': {'enabled': False, 'datasets': ['D1', 'D2']}}

In [2]:
rows = []
for dataset_id, info in cfg['paths'].items():
    manifest = repo_root / info['manifest']
    root = repo_root / info['root']
    row = {
        'dataset_id': dataset_id,
        'manifest': str(manifest.relative_to(repo_root)),
        'dataset_root': str(root.relative_to(repo_root)),
        'rows': 0,
        'columns': 'pending',
        'has_split_col': False,
        'non_empty_rows': 0,
        'status': 'pending',
    }

    if manifest.exists() and manifest.stat().st_size > 0:
        try:
            df = pd.read_csv(manifest)
            row.update({
                'rows': len(df),
                'columns': ', '.join(df.columns.tolist()) if len(df.columns) else 'pending',
                'has_split_col': 'split' in df.columns,
                'non_empty_rows': int(len(df.dropna(how='all'))),
                'status': 'available',
            })
        except pd.errors.EmptyDataError:
            row['status'] = 'empty_file_pending'

    rows.append(row)

manifest_summary = pd.DataFrame(rows)
manifest_summary

,dataset_id,manifest,dataset_root,rows,columns,has_split_col,non_empty_rows,status
0,D1,data/manifests/D1_tyrenet_manifest.csv,data/D1_tyrenet,1698,"image_path, label_str, label, dataset_id, split",True,1698,available
1,D2,data/manifests/D2_crack_manifest.csv,data/D2_tire_crack,0,pending,False,0,empty_file_pending
2,D3,data/manifests/D3_quality_manifest.csv,data/D3_tyre_quality,0,pending,False,0,empty_file_pending


## Interpretation
### Simple English
A populated manifest means the dataset is at least wired into the repo structure. An empty manifest means the dataset name exists in config, but the tracked repo does not yet provide usable rows for experiments.

### Technical detail
At the time of this notebook, D1 is the only tracked manifest with rows. D2 and D3 are configuration placeholders in the canonical dataset config, not completed tracked datasets in this checkout.

In [3]:
d1_manifest = repo_root / 'data/manifests/D1_tyrenet_manifest.csv'
d1 = pd.read_csv(d1_manifest)
print('D1 rows:', len(d1))
print('\nClass counts:')
print(d1['label_str'].value_counts(dropna=False))
print('\nSplit counts:')
print(d1.groupby(['split', 'label_str']).size().unstack(fill_value=0))

D1 rows: 1698

Class counts:
label_str
defect    866
good      832
Name: count, dtype: int64

Split counts:
label_str  defect  good
split                  
test          130   125
train         606   582
val           130   125


In [4]:
example_rows = d1.head(5).copy()
example_rows

,image_path,label_str,label,dataset_id,split
0,defect/Defective (422).jpg,defect,1,D1,train
1,good/good (456).jpg,good,0,D1,train
2,good/good (669).jpg,good,0,D1,train
3,good/good (783).jpg,good,0,D1,train
4,defect/Defective (764).jpg,defect,1,D1,train


## Path Resolution Check
### Simple English
The D1 manifest stores short paths like `defect/...` instead of full repo paths. That is okay because the active pipeline combines those paths with the dataset root from config.

### Technical detail
This is the normalized manifest convention adopted in the cleanup refactor. It is cleaner than mixing repo-relative and dataset-relative conventions.

In [5]:
dataset_root = repo_root / cfg['paths']['D1']['root']
resolved = dataset_root / d1.iloc[0]['image_path']
print('Resolved sample path:', resolved)
print('Exists locally:', resolved.exists())

Resolved sample path: /Users/ritik/Documents/Project TDA/TyreVisionX/data/D1_tyrenet/defect/Defective (422).jpg
Exists locally: False


## Pending Items
- D2 manifest population: pending
- D3 manifest population: pending
- Cross-dataset comparative analysis: pending until real rows and artifacts exist

## Future Work Sections
- Anomaly detection may need richer unlabeled or weakly labeled data.
- Localization and segmentation will require spatial annotations, not just image-level labels.
- Multi-view 3D work will require synchronized or related views of the same tyre.
- Knowledge reasoning would require a structured representation of defect types, causes, and visual evidence.